# Bangladesh Urban Center Mapping — ALL ONLINE

This notebook follows the **interactive Earth Search tutorial style**, but covers the **whole of Bangladesh** and implements the multi-factor urban-center workflow.

### Online sources
- **Sentinel-2:** Element84 Earth Search STAC
- **VIIRS Nighttime Lights:** Google Earth Engine
- **Population (2025):** GHSL population via Google Earth Engine
- **Topography:** Copernicus DEM GLO-30 via Google Earth Engine
- **Roads:** Overture Maps cloud GeoParquet via DuckDB
- **POI / Services:** Overture Maps Places cloud GeoParquet via DuckDB

No manual download of these factor datasets is required. Data are queried from the cloud inside VS Code/Jupyter. Only final outputs are written to your `outputs` folder.

> Earth Engine requires a one-time authentication and a Google Cloud project enabled for Earth Engine.


In [1]:
# 0. INSTALL PACKAGES (RUN ONCE IN A CLEAN CONDA ENVIRONMENT)
#
# Recommended:
#
# conda create -n urban_center -c conda-forge --override-channels ^
#   python=3.11 geopandas rasterio gdal proj pyproj rioxarray xarray dask ^
#   pystac-client stackstac leafmap shapely scipy matplotlib pandas numpy ^
#   pyogrio duckdb earthengine-api geemap xee ipykernel -y
#
# Then:
#
# conda activate urban_center
# python -m ipykernel install --user --name urban_center --display-name "Python (urban_center)"
#
# In VS Code select kernel: Python (urban_center)


In [2]:
# 1. CONFIGURATION
from pathlib import Path
import os

PROJECT_DIR = Path(r"E:\Geospatial\Urban Center\Urban-Center")
AOI_PATH = PROJECT_DIR / "bgd_admin_boundaries.shp" / "bgd_admin0.shp"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Earth Engine project:
# Replace this with YOUR Earth Engine-enabled Google Cloud project ID.
EE_PROJECT = "PUT_YOUR_EARTH_ENGINE_PROJECT_ID_HERE"

# Sentinel-2 time window
S2_DATE_RANGE = "2025-01-01/2025-03-31"
S2_START = "2025-01-01"
S2_END = "2025-04-01"
MAX_CLOUD = 10
N_BEST_PER_TILE = 5

# VIIRS persistent light year
VIIRS_START = "2025-01-01"
VIIRS_END = "2026-01-01"

# Analysis grid
TARGET_EPSG = 6933
RESOLUTION_M = 100
CHUNK_SIZE = 512

# Overture current stable release used in this notebook.
OVERTURE_RELEASE = "2026-06-17.0"

WEIGHTS = {
    "builtup": 0.30,
    "nightlight": 0.20,
    "population": 0.20,
    "road": 0.10,
    "poi": 0.10,
    "topography": 0.10,
}

URBAN_SCORE_THRESHOLD = 0.55
MIN_PATCH_AREA_KM2 = 1.0
MIN_MEAN_POP_DENSITY = 500.0
MIN_MEAN_URBAN_SCORE = 0.60

os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["GDAL_HTTP_RETRY_CODES"] = "ALL"
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"

print("AOI:", AOI_PATH)
print("Outputs:", OUTPUT_DIR)
print("Overture release:", OVERTURE_RELEASE)


AOI: E:\Geospatial\Urban Center\Urban-Center\bgd_admin_boundaries.shp\bgd_admin0.shp
Outputs: E:\Geospatial\Urban Center\Urban-Center\outputs
Overture release: 2026-06-17.0


In [3]:
# 2. IMPORTS + LOCAL GEO ENVIRONMENT CHECK
from collections import defaultdict, Counter
import warnings
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
from rasterio.errors import RasterioIOError
from rasterio.features import rasterize, shapes
from rasterio.enums import MergeAlg
import pyproj

import xarray as xr
import rioxarray
import stackstac
from pystac_client import Client

from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely import from_wkb

from scipy import ndimage
import leafmap
import duckdb

warnings.filterwarnings("ignore", category=FutureWarning)

print("Rasterio:", rasterio.__version__)
print("GDAL:", rasterio.__gdal_version__)
print("PROJ:", pyproj.proj_version_str)
print("StackSTAC:", stackstac.__version__)

# This catches the PROJ mismatch you previously encountered.
try:
    print("CRS test:", rasterio.crs.CRS.from_epsg(TARGET_EPSG))
except Exception as exc:
    raise RuntimeError(
        "Your GDAL/PROJ environment is broken. "
        "Create the clean 'urban_center' conda environment from Cell 0, "
        "then select that kernel in VS Code.\n\n"
        f"Original error: {exc}"
    )


Rasterio: 1.4.4
GDAL: 3.10.3
PROJ: 9.5.1
StackSTAC: 0.5.1


RuntimeError: Your GDAL/PROJ environment is broken. Create the clean 'urban_center' conda environment from Cell 0, then select that kernel in VS Code.

Original error: The EPSG code is unknown. PROJ: proj_create_from_database: C:\Users\HP\.conda\envs\geo\Library\share\proj\proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 5 is expected. It comes from another PROJ installation.

In [4]:
# 3. LOAD BANGLADESH AOI
if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"AOI not found:\n{AOI_PATH}\n"
        "Edit PROJECT_DIR / AOI_PATH in Cell 1."
    )

bd = gpd.read_file(AOI_PATH)

if bd.empty:
    raise ValueError("Bangladesh AOI is empty.")
if bd.crs is None:
    raise ValueError("Bangladesh AOI has no CRS.")

bd = bd.to_crs(4326)
aoi = bd[["geometry"]].dissolve().reset_index(drop=True)

if not aoi.geometry.iloc[0].is_valid:
    aoi["geometry"] = aoi.geometry.buffer(0)

bd_geom = aoi.geometry.iloc[0]
west, south, east, north = map(float, aoi.total_bounds)

print("CRS:", aoi.crs)
print("Bounds:", (west, south, east, north))
print("AOI valid:", bd_geom.is_valid)


CRS: EPSG:4326
Bounds: (88.00816912400006, 20.590608254000188, 92.68005782300008, 26.634548266000024)
AOI valid: True


In [5]:
# 4. SHOW WHOLE BANGLADESH IN LEAFMAP
m = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m.add_gdf(
    aoi,
    layer_name="Bangladesh AOI",
    style={
        "color": "red",
        "weight": 3,
        "fillColor": "red",
        "fillOpacity": 0.04,
    },
)

m


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

## Part A — Sentinel-2 from Earth Search

This keeps the same overall approach as the Earth Search tutorial, but replaces the single point with the **Bangladesh national bounding box**, followed by exact local intersection with the Bangladesh polygon.


In [6]:
# 5. SEARCH SENTINEL-2 — EARTH SEARCH
catalog = Client.open("https://earth-search.aws.element84.com/v1")

search = catalog.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=[west, south, east, north],
    datetime=S2_DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
)

items_raw = list(search.items())

print("Raw bbox results:", len(items_raw))

items_bd = []

for item in items_raw:
    if item.geometry is None:
        continue
    try:
        if shape(item.geometry).intersects(bd_geom):
            items_bd.append(item)
    except Exception as exc:
        print("Skipped:", item.id, type(exc).__name__)

if not items_bd:
    raise RuntimeError("No Sentinel-2 scenes intersect Bangladesh.")

dates = sorted({
    item.datetime.strftime("%Y-%m-%d")
    for item in items_bd
    if item.datetime is not None
})

print("Scenes intersecting Bangladesh:", len(items_bd))
print("Unique acquisition dates:", len(dates))
print("First dates:", dates[:10])


Raw bbox results: 984
Scenes intersecting Bangladesh: 634
Unique acquisition dates: 51
First dates: ['2025-01-02', '2025-01-03', '2025-01-05', '2025-01-07', '2025-01-10', '2025-01-12', '2025-01-13', '2025-01-15', '2025-01-17', '2025-01-18']


In [7]:
# 6. SHOW ALL SEARCHED SENTINEL-2 FOOTPRINTS
scene_gdf = gpd.GeoDataFrame(
    [
        {
            "id": item.id,
            "cloud": item.properties.get("eo:cloud_cover"),
            "geometry": shape(item.geometry),
        }
        for item in items_bd
    ],
    crs="EPSG:4326",
)

m_scenes = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m_scenes.add_gdf(
    scene_gdf,
    layer_name="Sentinel-2 footprints",
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.04,
    },
)

m_scenes.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={"color": "red", "weight": 3, "fillOpacity": 0},
)

m_scenes


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [8]:
# 7. SELECT LOW-CLOUD SCENES + GUARANTEE 100% FOOTPRINT COVERAGE

def get_tile_id(item):
    parts = item.id.split("_")
    if len(parts) > 1 and parts[1].startswith("T"):
        return parts[1]

    grid_code = item.properties.get("grid:code")
    if grid_code:
        return str(grid_code)

    raise ValueError(f"Cannot identify tile for {item.id}")


items_by_tile = defaultdict(list)

for item in items_bd:
    try:
        items_by_tile[get_tile_id(item)].append(item)
    except ValueError as exc:
        print(exc)


selected_items = []

for tile_id, tile_items in items_by_tile.items():
    tile_items = sorted(
        tile_items,
        key=lambda item: (
            item.properties.get("eo:cloud_cover", 100),
            item.datetime.isoformat() if item.datetime else "",
        ),
    )
    selected_items.extend(tile_items[:N_BEST_PER_TILE])


def union_footprints(stac_items):
    geoms = [
        shape(item.geometry)
        for item in stac_items
        if item.geometry is not None
    ]
    return unary_union(geoms)


selected_union = union_footprints(selected_items)
missing_geom = bd_geom.difference(selected_union)

selected_ids = {item.id for item in selected_items}

remaining_items = sorted(
    [item for item in items_bd if item.id not in selected_ids],
    key=lambda item: item.properties.get("eo:cloud_cover", 100),
)

extra_items = []

for item in remaining_items:
    if missing_geom.is_empty:
        break

    scene_geom = shape(item.geometry)
    gain = missing_geom.intersection(scene_geom)

    if not gain.is_empty and gain.area > 0:
        selected_items.append(item)
        extra_items.append(item)
        selected_union = selected_union.union(scene_geom)
        missing_geom = bd_geom.difference(selected_union)


coverage_check = gpd.GeoDataFrame(
    {"kind": ["aoi", "covered"]},
    geometry=[
        bd_geom,
        bd_geom.intersection(selected_union),
    ],
    crs="EPSG:4326",
).to_crs(TARGET_EPSG)

coverage_pct = (
    coverage_check.geometry.iloc[1].area
    / coverage_check.geometry.iloc[0].area
    * 100
)

print("Unique MGRS tiles:", len(items_by_tile))
print("Selected scenes:", len(selected_items))
print("Extra scenes added:", len(extra_items))
print(f"Bangladesh footprint coverage: {coverage_pct:.6f}%")


Unique MGRS tiles: 36
Selected scenes: 187
Extra scenes added: 7
Bangladesh footprint coverage: 100.000000%


In [9]:
# 8. SHOW FINAL SELECTED FOOTPRINTS
selected_scene_gdf = gpd.GeoDataFrame(
    [
        {
            "id": item.id,
            "cloud": item.properties.get("eo:cloud_cover"),
            "geometry": shape(item.geometry),
        }
        for item in selected_items
    ],
    crs="EPSG:4326",
)

m_selected = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m_selected.add_gdf(
    selected_scene_gdf,
    layer_name="Selected scenes",
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.08,
    },
)

m_selected.add_gdf(
    aoi,
    layer_name="Bangladesh boundary",
    style={"color": "red", "weight": 3, "fillOpacity": 0},
)

m_selected


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [10]:
# 9. BUILD ONE RESILIENT SENTINEL-2 STACK
REQUIRED_ASSETS = ["blue", "green", "red", "nir", "swir16", "scl"]

selected_items_clean = [
    item
    for item in selected_items
    if all(asset in item.assets for asset in REQUIRED_ASSETS)
]

print("Scenes with all required assets:", len(selected_items_clean))
print(
    "Dropped for missing asset metadata:",
    len(selected_items) - len(selected_items_clean),
)

if not selected_items_clean:
    raise RuntimeError("No selected scene has all required bands.")

sentinel = stackstac.stack(
    selected_items_clean,
    assets=REQUIRED_ASSETS,
    bounds_latlon=[west, south, east, north],
    epsg=TARGET_EPSG,
    resolution=RESOLUTION_M,
    chunksize=CHUNK_SIZE,
    dtype=np.float32,
    fill_value=np.float32(np.nan),
    rescale=False,
    errors_as_nodata=(RasterioIOError(r".*"),),
)

print(sentinel)
print("Virtual stack shape:", sentinel.shape)


Scenes with all required assets: 187
Dropped for missing asset metadata: 0
<xarray.DataArray 'stackstac-70774311adfebdce4b4b7202b37e8630' (time: 187,
                                                                band: 6,
                                                                y: 7078, x: 4509)> Size: 143GB
dask.array<fetch_raster_window, shape=(187, 6, 7078, 4509), dtype=float32, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>
Coordinates: (12/54)
  * time                                     (time) datetime64[ns] 1kB 2025-0...
    id                                       (time) <U30 22kB 'S2A_T45RYK_202...
  * band                                     (band) <U6 144B 'blue' ... 'scl'
  * x                                        (x) float64 36kB 8.492e+06 ... 8...
  * y                                        (y) float64 57kB 3.28e+06 ... 2....
    s2:datastrip_id                          (time) <U64 48kB 'S2A_OPER_MSI_L...
    ...                                       ...


In [11]:
# 10. SMALL REAL-READ TEST
cx = sentinel.sizes["x"] // 2
cy = sentinel.sizes["y"] // 2

test_stack = sentinel.isel(
    time=slice(0, min(3, sentinel.sizes["time"])),
    x=slice(max(0, cx - 128), min(cx + 128, sentinel.sizes["x"])),
    y=slice(max(0, cy - 128), min(cy + 128, sentinel.sizes["y"])),
).compute()

print("Small remote-read test successful.")
print("Shape:", test_stack.shape)
print("Finite values:", int(np.isfinite(test_stack.values).sum()))


Small remote-read test successful.
Shape: (3, 6, 256, 256)
Finite values: 0


In [12]:
# 11. CLOUD MASK + TEMPORAL MEDIAN
scl = sentinel.sel(band="scl")

INVALID_SCL = [0, 1, 3, 8, 9, 10, 11]
valid = ~scl.isin(INVALID_SCL)

blue_med = sentinel.sel(band="blue").where(valid).median("time", skipna=True)
green_med = sentinel.sel(band="green").where(valid).median("time", skipna=True)
red_med = sentinel.sel(band="red").where(valid).median("time", skipna=True)
nir_med = sentinel.sel(band="nir").where(valid).median("time", skipna=True)
swir_med = sentinel.sel(band="swir16").where(valid).median("time", skipna=True)

print("Cloud-masked median composites prepared.")


Cloud-masked median composites prepared.


In [13]:
# 12. SENTINEL-2 INDICES
EPS = np.float32(1e-6)

def safe_nd(a, b):
    denominator = a + b
    return (
        (a - b) / denominator.where(np.abs(denominator) > EPS)
    ).clip(-1, 1).astype("float32")

ndbi = safe_nd(swir_med, nir_med).rename("NDBI")
ndvi = safe_nd(nir_med, red_med).rename("NDVI")
mndwi = safe_nd(green_med, swir_med).rename("MNDWI")

bsi_num = (swir_med + red_med) - (nir_med + blue_med)
bsi_den = (swir_med + red_med) + (nir_med + blue_med)

bsi = (
    bsi_num / bsi_den.where(np.abs(bsi_den) > EPS)
).clip(-1, 1).astype("float32").rename("BSI")

print("Prepared: NDBI, NDVI, MNDWI, BSI")


Prepared: NDBI, NDVI, MNDWI, BSI


In [14]:
# 13. BUILT-UP PROBABILITY (0–1)
# This is an interpretable score, not a pretrained classifier.

def index_to_01(da):
    return ((da + 1.0) / 2.0).clip(0, 1).astype("float32")

ndbi01 = index_to_01(ndbi)
ndvi01 = index_to_01(ndvi)
mndwi01 = index_to_01(mndwi)
bsi01 = index_to_01(bsi)

builtup_probability = (
    0.50 * ndbi01
    + 0.20 * (1.0 - ndvi01)
    + 0.20 * (1.0 - mndwi01)
    + 0.10 * (1.0 - bsi01)
).clip(0, 1).astype("float32").rename("Builtup_Probability")

print("Built-up probability READY.")


Built-up probability READY.


In [16]:
# 14. CREATE COMMON 100-m TEMPLATE + EXACT BANGLADESH CLIP
template = (
    ndbi.astype("float32")
    .rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
)

aoi_target = aoi.to_crs(TARGET_EPSG)

def add_rio_metadata(da):
    return (
        da.rio
        .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )

def clip_bd(da):
    da = add_rio_metadata(da)
    return da.rio.clip(
        aoi_target.geometry,
        aoi_target.crs,
        drop=True,
        all_touched=False,
    )

def align_to_template(da, resampling="bilinear"):
    import rasterio.enums
    method = {
        "nearest": rasterio.enums.Resampling.nearest,
        "bilinear": rasterio.enums.Resampling.bilinear,
    }[resampling]

    return da.rio.reproject_match(
        template,
        resampling=method,
    ).astype("float32")

print("Template CRS:", template.rio.crs)
print("Template resolution:", template.rio.resolution())


Template CRS: PROJCS["WGS 84 / NSIDC EASE-Grid 2.0 Global",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","6933"]]
Template resolution: (100.0, -100.0)


## Part B — Online VIIRS, Population and Topography via Earth Engine + Xee

Xee lets Earth Engine imagery appear as lazy `xarray` objects in Python/VS Code. There is no manual raster download step.


In [17]:
%pip install xee

Note: you may need to restart the kernel to use updated packages.


In [18]:
# 15. EARTH ENGINE AUTHENTICATION / INITIALIZATION

import ee

EE_PROJECT = "solid-garden-458417-t4"

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized successfully.")

except Exception:
    print("Authentication required...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized successfully after authentication.")

# Bangladesh geometry from your already-loaded shapefile
ee_aoi = ee.Geometry(bd_geom.__geo_interface__)

print("Bangladesh AOI transferred to Earth Engine.")
print("EE Project:", EE_PROJECT)

Earth Engine initialized successfully.
Bangladesh AOI transferred to Earth Engine.
EE Project: solid-garden-458417-t4


In [19]:
# 16. HELPER: EARTH ENGINE IMAGE -> XARRAY ON 100-m BANGLADESH GRID

def ee_image_to_xarray(image, band_name, output_name):
    """
    Read one processed Earth Engine image lazily through Xee.
    The source is cloud-hosted; no manual input raster is needed.
    """
    image = ee.Image(image).select([band_name]).rename([output_name])

    # Fit the Bangladesh bounding box to a 100-m grid in EPSG:6933.
    grid = helpers.fit_geometry(
        geometry=bd_geom,
        geometry_crs="EPSG:4326",
        grid_crs=f"EPSG:{TARGET_EPSG}",
        grid_scale=(RESOLUTION_M, -RESOLUTION_M),
    )

    ds = xr.open_dataset(
        ee.ImageCollection([image]),
        engine="ee",
        **grid,
    )

    da = ds[output_name]

    if "time" in da.dims:
        da = da.isel(time=0, drop=True)

    # Xee uses x/y coordinates; attach CRS and match exact Sentinel grid.
    da = (
        da.astype("float32")
        .rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )

    return align_to_template(da, "bilinear")


def ee_percentile_norm(image, band, region, scale):
    """
    Robust 2nd–98th percentile normalization performed server-side.
    """
    image = ee.Image(image).select(band)

    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )

    lo = ee.Number(stats.get(f"{band}_p2"))
    hi = ee.Number(stats.get(f"{band}_p98"))

    return (
        image.subtract(lo)
        .divide(hi.subtract(lo).max(1e-6))
        .clamp(0, 1)
    )


In [23]:
# 16. EARTH ENGINE IMAGE -> XARRAY
# No xee.helpers used

def ee_image_to_xarray(image, band_name, output_name):

    image = (
        ee.Image(image)
        .select([band_name])
        .rename([output_name])
    )

    collection = ee.ImageCollection([image])

    ds = xr.open_dataset(
        collection,
        engine="ee",
        geometry=bd_geom.__geo_interface__,
        crs=f"EPSG:{TARGET_EPSG}",
        scale=RESOLUTION_M,
    )

    da = ds[output_name]

    if "time" in da.dims:
        da = da.isel(time=0, drop=True)

    da = da.astype("float32")

    da = (
        da.rio
        .set_spatial_dims(
            x_dim="x",
            y_dim="y",
            inplace=False
        )
        .rio.write_crs(
            f"EPSG:{TARGET_EPSG}",
            inplace=False
        )
    )

    da = align_to_template(
        da,
        "bilinear"
    )

    return da


def ee_percentile_norm(
    image,
    band,
    region,
    scale
):

    image = ee.Image(image).select(band)

    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )

    lo = ee.Number(
        stats.get(f"{band}_p2")
    )

    hi = ee.Number(
        stats.get(f"{band}_p98")
    )

    normalized = (
        image
        .subtract(lo)
        .divide(
            hi.subtract(lo).max(1e-6)
        )
        .clamp(0, 1)
    )

    return normalized


print("EE helper functions updated successfully.")

EE helper functions updated successfully.


In [27]:
pip install --upgrade --pre xee

Note: you may need to restart the kernel to use updated packages.


In [28]:
import xee
print(xee.__version__)

0.1.2


In [29]:
# ============================================================
# CELL 16 — CORRECT XEE GRID HELPER
# ============================================================

import ee
import xarray as xr
import rioxarray
from xee import helpers


def ee_image_to_xarray(image, band_name, output_name):

    image = (
        ee.Image(image)
        .select([band_name])
        .rename([output_name])
    )

    collection = ee.ImageCollection([image])

    # Build output grid for whole Bangladesh
    grid = helpers.fit_geometry(
        geometry=bd_geom,
        geometry_crs="EPSG:4326",
        grid_crs=f"EPSG:{TARGET_EPSG}",
        grid_scale=(RESOLUTION_M, -RESOLUTION_M),
    )

    ds = xr.open_dataset(
        collection,
        engine="ee",
        **grid,
    )

    da = ds[output_name]

    if "time" in da.dims:
        da = da.isel(time=0, drop=True)

    da = (
        da.astype("float32")
        .rio.set_spatial_dims(
            x_dim="x",
            y_dim="y",
            inplace=False,
        )
        .rio.write_crs(
            f"EPSG:{TARGET_EPSG}",
            inplace=False,
        )
    )

    # Align with Sentinel grid
    da = align_to_template(
        da,
        "bilinear",
    )

    return da


def ee_percentile_norm(
    image,
    band,
    region,
    scale,
):

    image = ee.Image(image).select(band)

    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )

    lo = ee.Number(
        stats.get(f"{band}_p2")
    )

    hi = ee.Number(
        stats.get(f"{band}_p98")
    )

    return (
        image
        .subtract(lo)
        .divide(
            hi.subtract(lo).max(1e-6)
        )
        .clamp(0, 1)
    )


print("Xee helper functions READY")

Xee helper functions READY


In [ ]:
# ============================================================
# VIIRS NIGHTTIME LIGHT — WORLD BANK OPEN NIGHT LIGHTS STYLE
# BANGLADESH 2025
# ============================================================

import ee
import geemap


# ------------------------------------------------------------
# 1. EARTH ENGINE INITIALIZATION
# ------------------------------------------------------------

EE_PROJECT = "solid-garden-458417-t4"

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized.")

except Exception:

    ee.Authenticate()

    ee.Initialize(project=EE_PROJECT)

    print("Earth Engine initialized after authentication.")


# ------------------------------------------------------------
# 2. BANGLADESH AOI
# ------------------------------------------------------------
# bd_geom already comes from your Bangladesh shapefile

ee_aoi = ee.Geometry(
    bd_geom.__geo_interface__
)

print("Bangladesh AOI ready.")


# ------------------------------------------------------------
# 3. VIIRS MONTHLY COLLECTION — 2025
# ------------------------------------------------------------

viirs_2025 = (
    ee.ImageCollection(
        "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"
    )

    .filterDate(
        "2025-01-01",
        "2026-01-01"
    )

    .filterBounds(
        ee_aoi
    )

    .select(
        "avg_rad"
    )
)


print(
    "VIIRS monthly images:",
    viirs_2025.size().getInfo()
)


# ------------------------------------------------------------
# 4. CLIP EVERY MONTH TO BANGLADESH
# ------------------------------------------------------------

viirs_bd = viirs_2025.map(
    lambda image:
        image
        .clip(ee_aoi)
        .copyProperties(
            image,
            ["system:time_start"]
        )
)


# ------------------------------------------------------------
# 5. ANNUAL MEDIAN COMPOSITE
# ------------------------------------------------------------
# Same basic logic as World Bank Open Night Lights tutorial

viirs_annual = (
    viirs_bd
    .median()
    .rename(
        "VIIRS_2025"
    )
)


print(
    "Annual VIIRS composite READY."
)


# ------------------------------------------------------------
# 6. MASK ZERO / NEGATIVE VALUES
# ------------------------------------------------------------

viirs_annual = (
    viirs_annual
    .updateMask(
        viirs_annual.gt(0)
    )
)


# ------------------------------------------------------------
# 7. VISUALIZATION PARAMETERS
# ------------------------------------------------------------

viirs_vis = {
    "min": 0,
    "max": 20,
    "palette": [
        "000000",
        "0d0887",
        "7e03a8",
        "cc4778",
        "f89540",
        "f0f921"
    ]
}


# ------------------------------------------------------------
# 8. SHOW IN VS CODE
# ------------------------------------------------------------

Map = geemap.Map(
    center=[
        23.6850,
        90.3563
    ],
    zoom=7,
    height="750px"
)


Map.add_basemap(
    "SATELLITE"
)


Map.addLayer(
    viirs_annual,
    viirs_vis,
    "VIIRS-DNB Bangladesh 2025"
)


# Bangladesh boundary
boundary = ee.Image().byte().paint(
    featureCollection=ee.FeatureCollection(
        [
            ee.Feature(
                ee_aoi
            )
        ]
    ),
    color=1,
    width=2
)


Map.addLayer(
    boundary,
    {
        "palette": ["red"]
    },
    "Bangladesh Boundary"
)


Map.addLayerControl()

Map

Earth Engine initialized.
Bangladesh AOI ready.
VIIRS monthly images: 12
Annual VIIRS composite READY.


Map(center=[23.685, 90.3563], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

AttributeError: 'method_descriptor' object has no attribute 'today'

AttributeError: 'method' object has no attribute 'markers'

In [36]:
# ============================================================
# GHSL POPULATION 2025 — EARTH ENGINE NATIVE
# NO XEE
# ============================================================

import ee
import geemap


# ------------------------------------------------------------
# 1. LOAD GHSL 2025
# ------------------------------------------------------------

ghsl_col = (
    ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
    .filterDate("2025-01-01", "2026-01-01")
    .filterBounds(ee_aoi)
)

ghsl_n = ghsl_col.size().getInfo()

print("GHSL 2025 images:", ghsl_n)

if ghsl_n == 0:
    raise RuntimeError(
        "No GHSL population image found for 2025."
    )


# ------------------------------------------------------------
# 2. POPULATION COUNT
# ------------------------------------------------------------

pop_count_ee = (
    ee.Image(ghsl_col.first())
    .select("population_count")
    .max(0)
    .clip(ee_aoi)
    .rename("population_count")
)


# ------------------------------------------------------------
# 3. POPULATION DENSITY
# ------------------------------------------------------------

pop_density_ee = (
    pop_count_ee
    .multiply(100.0)
    .rename("population_density")
)


# ------------------------------------------------------------
# 4. LOG TRANSFORMATION
# ------------------------------------------------------------

pop_log = (
    pop_density_ee
    .add(1)
    .log()
    .rename("pop_log")
)


# ------------------------------------------------------------
# 5. 0–1 POPULATION SCORE
# ------------------------------------------------------------

population_score_ee = (
    ee_percentile_norm(
        pop_log,
        "pop_log",
        ee_aoi,
        100
    )
    .rename("Population_Score")
    .clip(ee_aoi)
)


# ------------------------------------------------------------
# 6. CHECK
# ------------------------------------------------------------

print("population : READY")

print(
    "Population bands:",
    population_score_ee.bandNames().getInfo()
)

GHSL 2025 images: 1
population : READY
Population bands: ['Population_Score']


In [37]:
# ============================================================
# SHOW GHSL POPULATION SCORE
# ============================================================

pop_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "ffffcc",
        "c2e699",
        "78c679",
        "31a354",
        "006837"
    ]
}

Map_pop = geemap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="750px"
)

Map_pop.add_basemap("SATELLITE")

Map_pop.addLayer(
    population_score_ee,
    pop_vis,
    "GHSL Population Score 2025"
)

Map_pop.addLayer(
    ee.Image().byte().paint(
        ee.FeatureCollection([
            ee.Feature(ee_aoi)
        ]),
        1,
        2
    ),
    {"palette": ["red"]},
    "Bangladesh Boundary"
)

Map_pop.addLayerControl()

Map_pop

Map(center=[23.685, 90.3563], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [39]:
# ============================================================
# 21. COPERNICUS DEM + SLOPE — ONLINE
# EARTH ENGINE NATIVE
# NO XEE
# ============================================================

import ee
import geemap


# ------------------------------------------------------------
# 1. LOAD COPERNICUS DEM
# ------------------------------------------------------------

dem_col = ee.ImageCollection(
    "COPERNICUS/DEM/GLO30_2024_1"
)

native_projection = (
    dem_col
    .first()
    .select("DEM")
    .projection()
)


# ------------------------------------------------------------
# 2. MOSAIC DEM
# ------------------------------------------------------------

dem_ee = (
    dem_col
    .select("DEM")
    .mosaic()
    .setDefaultProjection(
        native_projection
    )
    .clip(
        ee_aoi
    )
    .rename(
        "elevation"
    )
)


print(
    "DEM ready."
)


# ------------------------------------------------------------
# 3. SLOPE
# ------------------------------------------------------------

slope_ee = (
    ee.Terrain
    .slope(
        dem_ee
    )
    .clip(
        ee_aoi
    )
    .rename(
        "slope"
    )
)


print(
    "Slope ready."
)


# ------------------------------------------------------------
# 4. NORMALIZE ELEVATION
# ------------------------------------------------------------

elev01_ee = (
    ee_percentile_norm(
        dem_ee,
        "elevation",
        ee_aoi,
        100
    )
    .rename(
        "elevation01"
    )
)


# ------------------------------------------------------------
# 5. NORMALIZE SLOPE
# ------------------------------------------------------------

slope01_ee = (
    ee_percentile_norm(
        slope_ee,
        "slope",
        ee_aoi,
        100
    )
    .rename(
        "slope01"
    )
)


# ------------------------------------------------------------
# 6. TOPOGRAPHY SCORE
# ------------------------------------------------------------
#
# Lower elevation = higher score
# Lower slope     = higher score
#
# Weight:
# elevation = 0.50
# slope     = 0.50
#

topography_ee = (
    ee.Image(1)
    .subtract(
        elev01_ee
    )
    .multiply(
        0.50
    )

    .add(

        ee.Image(1)
        .subtract(
            slope01_ee
        )
        .multiply(
            0.50
        )

    )

    .clamp(
        0,
        1
    )

    .clip(
        ee_aoi
    )

    .rename(
        "Topography_Score"
    )
)


# ------------------------------------------------------------
# 7. CHECK
# ------------------------------------------------------------

print(
    "Topography bands:",
    topography_ee.bandNames().getInfo()
)

print()
print(
    "topography  : READY"
)

DEM ready.
Slope ready.
Topography bands: ['Topography_Score']

topography  : READY


In [40]:
# ============================================================
# 22. SHOW DEM / SLOPE / TOPOGRAPHY SCORE
# ============================================================

Map_topo = geemap.Map(
    center=[
        23.6850,
        90.3563
    ],
    zoom=7,
    height="750px"
)


# ------------------------------------------------------------
# DEM
# ------------------------------------------------------------

dem_vis = {
    "min": 0,
    "max": 100,
    "palette": [
        "006837",
        "31a354",
        "78c679",
        "c2e699",
        "ffffcc",
        "fed976",
        "fd8d3c",
        "bd0026"
    ]
}


Map_topo.addLayer(
    dem_ee,
    dem_vis,
    "Elevation"
)


# ------------------------------------------------------------
# SLOPE
# ------------------------------------------------------------

slope_vis = {
    "min": 0,
    "max": 30,
    "palette": [
        "ffffcc",
        "a1dab4",
        "41b6c4",
        "2c7fb8",
        "253494"
    ]
}


Map_topo.addLayer(
    slope_ee,
    slope_vis,
    "Slope"
)


# ------------------------------------------------------------
# TOPOGRAPHY SCORE
# ------------------------------------------------------------

topo_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "d73027",
        "fc8d59",
        "fee08b",
        "d9ef8b",
        "91cf60",
        "1a9850"
    ]
}


Map_topo.addLayer(
    topography_ee,
    topo_vis,
    "Topography Score"
)


# ------------------------------------------------------------
# BANGLADESH BOUNDARY
# ------------------------------------------------------------

boundary = (
    ee.Image()
    .byte()
    .paint(
        ee.FeatureCollection(
            [
                ee.Feature(
                    ee_aoi
                )
            ]
        ),
        1,
        2
    )
)


Map_topo.addLayer(
    boundary,
    {
        "palette": [
            "red"
        ]
    },
    "Bangladesh Boundary"
)


Map_topo.addLayerControl()

Map_topo


Map(center=[23.685, 90.3563], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

## Part C — Roads + POI directly from Overture Maps cloud

DuckDB reads the public Overture GeoParquet files directly from Amazon S3 and applies the Bangladesh bounding-box filter in the cloud query. You do not have to manually download a Bangladesh roads or POI file.


In [41]:
# 23. CONNECT DUCKDB TO OVERTURE CLOUD
con = duckdb.connect(database=":memory:")

# DuckDB extensions.
con.execute("INSTALL spatial")
con.execute("LOAD spatial")
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("SET s3_region='us-west-2'")
con.execute("SET enable_object_cache=true")
con.execute("SET threads=4")

ROAD_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=transportation/type=segment/*"
)

POI_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=places/type=place/*"
)

print("DuckDB + Overture cloud READY")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB + Overture cloud READY


In [48]:
import duckdb
import geopandas as gpd
import numpy as np
import xarray as xr

from shapely import from_wkb
from shapely.geometry import Point

from rasterio.features import rasterize
from rasterio.enums import MergeAlg
from scipy import ndimage

OVERTURE_RELEASE = "2026-08-19.0"

ROAD_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=transportation/type=segment/*"
)

print("Overture release:", OVERTURE_RELEASE)
print("ROAD_URL:", ROAD_URL)

Overture release: 2026-08-19.0
ROAD_URL: s3://overturemaps-us-west-2/release/2026-08-19.0/theme=transportation/type=segment/*


In [49]:
# STEP 2 — DUCKDB CONNECTION

import duckdb

con = duckdb.connect(database=":memory:")

con.execute("INSTALL spatial")
con.execute("LOAD spatial")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("SET s3_region='us-west-2'")
con.execute("SET enable_object_cache=true")
con.execute("SET threads=4")

print("DuckDB ready.")

DuckDB ready.


In [50]:
# STEP 3 — CHECK BANGLADESH AOI VARIABLES

print("west :", west)
print("south:", south)
print("east :", east)
print("north:", north)

print("AOI CRS:", aoi.crs)

west : 88.00816912400006
south: 20.590608254000188
east : 92.68005782300008
north: 26.634548266000024
AOI CRS: EPSG:4326


In [51]:
# STEP 4 — QUERY BANGLADESH ROADS ONLINE

road_sql = f"""
SELECT
    id,
    subtype,
    class,
    ST_AsWKB(geometry) AS geom_wkb

FROM read_parquet(
    '{ROAD_URL}',
    hive_partitioning=1
)

WHERE
    subtype = 'road'

    AND class IN (
        'motorway',
        'trunk',
        'primary',
        'secondary',
        'tertiary',
        'residential',
        'living_street',
        'unclassified',
        'service'
    )

    AND bbox.xmin <= {east}
    AND bbox.xmax >= {west}
    AND bbox.ymin <= {north}
    AND bbox.ymax >= {south}
"""

print("Querying Overture roads online...")

roads_df = con.execute(
    road_sql
).fetch_df()

print("Raw road records:", len(roads_df))

Querying Overture roads online...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw road records: 1722505


In [53]:
# ============================================================
# STEP 5 — CONVERT OVERTURE ROADS TO GEODATAFRAME
# FIX: bytearray -> bytes
# ============================================================

import geopandas as gpd
from shapely import from_wkb

print("Converting road geometries...")

if roads_df.empty:
    raise RuntimeError("roads_df is empty.")


# ------------------------------------------------------------
# 1. bytearray -> bytes
# ------------------------------------------------------------

wkb_values = roads_df["geom_wkb"].apply(
    lambda x: bytes(x) if isinstance(x, bytearray) else x
)


# ------------------------------------------------------------
# 2. WKB -> Shapely geometry
# ------------------------------------------------------------

road_geometry = from_wkb(
    wkb_values.to_numpy()
)


# ------------------------------------------------------------
# 3. GeoDataFrame
# ------------------------------------------------------------

roads = gpd.GeoDataFrame(
    roads_df.drop(
        columns=["geom_wkb"]
    ),
    geometry=road_geometry,
    crs="EPSG:4326"
)


# ------------------------------------------------------------
# 4. Remove null / empty geometry
# ------------------------------------------------------------

roads = roads[
    roads.geometry.notna()
    &
    ~roads.geometry.is_empty
].copy()


# ------------------------------------------------------------
# 5. Check
# ------------------------------------------------------------

print()
print("=" * 55)
print("Road GeoDataFrame : READY")
print("Road segments     :", len(roads))
print("CRS               :", roads.crs)
print("=" * 55)

display(
    roads.head()
)

Converting road geometries...

Road GeoDataFrame : READY
Road segments     : 1722505
CRS               : EPSG:4326


,id,subtype,class,geometry
0,cfe24b91-f0ee-4104-8a8f-1bbdb6028500,road,tertiary,"LINESTRING (87.97007 22.02874, 87.97045 22.028..."
1,4ca0ef24-ab02-4aec-bbb3-e97a5070bb97,road,tertiary,"LINESTRING (88.01084 22.00314, 88.01092 22.003..."
2,244b465f-6027-473e-aa7b-7855519fa66a,road,unclassified,"LINESTRING (87.99224 22.06407, 87.99229 22.064..."
3,e2f591e1-4894-48bf-8af5-f3aab02a9985,road,unclassified,"LINESTRING (88.01228 22.0586, 88.0123 22.05871..."
4,365926ce-569f-45a7-9d7b-d18ece9a5606,road,unclassified,"LINESTRING (88.01427 22.03773, 88.01359 22.038..."


In [ ]:
# ============================================================
# STEP 6 — FAST BANGLADESH ROAD FILTER
# DO NOT USE gpd.clip() ON 1.7 MILLION FEATURES
# ============================================================

print("Road segments before filter:", len(roads))

# Bangladesh polygon
bd_polygon = aoi.geometry.union_all()

print("Filtering roads by Bangladesh intersection...")

# Spatial-index-assisted filtering
roads_bd = roads[
    roads.geometry.intersects(bd_polygon)
].copy()

# Remove empty geometries
roads_bd = roads_bd[
    roads_bd.geometry.notna()
    &
    ~roads_bd.geometry.is_empty
].copy()

print()
print("=" * 55)
print("BANGLADESH ROAD FILTER : READY")
print("Before :", len(roads))
print("After  :", len(roads_bd))
print("CRS    :", roads_bd.crs)
print("=" * 55)

display(roads_bd.head())

In [ ]:
# 25. SHOW OVERTURE ROADS IN VS CODE
#
# For notebook responsiveness, display only a sample if there are many roads.

roads_display = roads

if len(roads_display) > 50000:
    roads_display = roads_display.sample(
        50000,
        random_state=42,
    )

m_roads = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m_roads.add_gdf(
    roads_display,
    layer_name="Overture Roads (display sample)",
    style={"color": "blue", "weight": 1},
)

m_roads.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={"color": "red", "weight": 2, "fillOpacity": 0},
)

m_roads


In [ ]:
# 26. ROAD DENSITY + INTERSECTION DENSITY -> ROAD SCORE

roads_m = roads.to_crs(TARGET_EPSG)

# Remove empty geometry.
roads_m = roads_m[
    roads_m.geometry.notna()
    & ~roads_m.geometry.is_empty
].copy()

transform = template.rio.transform()
out_shape = (
    template.sizes["y"],
    template.sizes["x"],
)

# 1) Road-presence raster
road_presence = rasterize(
    [(geom, 1) for geom in roads_m.geometry],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="uint8",
    all_touched=True,
)

# Approx. 1 km x 1 km window at 100 m.
window_cells = max(
    1,
    int(round(1000 / RESOLUTION_M)),
)

road_density_np = ndimage.uniform_filter(
    road_presence.astype("float32"),
    size=window_cells,
    mode="constant",
    cval=0,
)

# 2) Intersection proxy:
# Count repeated road endpoints after rounding projected coordinates.
endpoint_keys = []

for geom in roads_m.geometry:
    try:
        coords = list(geom.coords)
        if len(coords) >= 2:
            for xy in (coords[0], coords[-1]):
                endpoint_keys.append(
                    (
                        round(xy[0], 0),
                        round(xy[1], 0),
                    )
                )
    except Exception:
        pass

endpoint_counts = Counter(endpoint_keys)

intersection_points = [
    Point(x, y)
    for (x, y), count in endpoint_counts.items()
    if count >= 3
]

intersection_raster = rasterize(
    [(geom, 1) for geom in intersection_points],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="float32",
    merge_alg=MergeAlg.add,
)

intersection_density_np = ndimage.uniform_filter(
    intersection_raster,
    size=window_cells,
    mode="constant",
    cval=0,
)

def np_to_template(arr, name):
    da = xr.DataArray(
        arr.astype("float32"),
        dims=("y", "x"),
        coords={
            "y": template.y.values,
            "x": template.x.values,
        },
        name=name,
    )
    return add_rio_metadata(da)

road_density = np_to_template(
    road_density_np,
    "Road_Density",
)

intersection_density = np_to_template(
    intersection_density_np,
    "Intersection_Density",
)

def robust_norm_local(da):
    arr = da.values
    vals = arr[np.isfinite(arr)]

    if vals.size == 0:
        raise ValueError("No finite data for normalization.")

    lo, hi = np.nanpercentile(vals, [2, 98])

    if hi <= lo:
        return xr.zeros_like(da, dtype="float32")

    return (
        (da - lo) / (hi - lo)
    ).clip(0, 1).astype("float32")

road_density01 = robust_norm_local(road_density)
intersection_density01 = robust_norm_local(intersection_density)

road_score = (
    0.70 * road_density01
    + 0.30 * intersection_density01
).clip(0, 1).astype("float32").rename("Road_Score")

print("Intersections:", len(intersection_points))
print("road        : READY")


In [ ]:
# 27. QUERY BANGLADESH POI / SERVICES DIRECTLY FROM OVERTURE CLOUD
#
# We use basic_category / taxonomy text to select urban-service POIs:
# market/shopping, education, health, commercial/services.

poi_sql = f"""
SELECT
    id,
    basic_category,
    CAST(taxonomy AS VARCHAR) AS taxonomy_text,
    ST_AsWKB(geometry) AS geom_wkb
FROM read_parquet(
    '{POI_URL}',
    hive_partitioning=1
)
WHERE
    bbox.xmin BETWEEN {west} AND {east}
    AND bbox.ymin BETWEEN {south} AND {north}
    AND (
        regexp_matches(
            lower(coalesce(basic_category, '')),
            'market|shop|mall|grocery|supermarket|school|college|university|hospital|clinic|medical|bank|office|restaurant|hotel|pharmacy|service'
        )
        OR regexp_matches(
            lower(coalesce(CAST(taxonomy AS VARCHAR), '')),
            'market|shopping|school|education|college|university|hospital|clinic|health|medical|commercial|business|bank|office|restaurant|hotel|pharmacy|service'
        )
    )
"""

print("Querying Overture POIs online...")
poi_df = con.execute(poi_sql).fetch_df()

if poi_df.empty:
    raise RuntimeError("Overture POI query returned no records.")

poi_geoms = from_wkb(poi_df.pop("geom_wkb").to_numpy())

pois = gpd.GeoDataFrame(
    poi_df,
    geometry=poi_geoms,
    crs="EPSG:4326",
)

pois = gpd.clip(pois, aoi)

print("Urban-service POIs in Bangladesh:", len(pois))
print(pois.head())


In [ ]:
# 28. CLASSIFY POIs INTO THE WORKFLOW'S FOUR GROUPS

def classify_poi(row):
    text = (
        str(row.get("basic_category", ""))
        + " "
        + str(row.get("taxonomy_text", ""))
    ).lower()

    if any(k in text for k in [
        "school", "college", "university", "education",
        "academy", "kindergarten",
    ]):
        return "education"

    if any(k in text for k in [
        "hospital", "clinic", "medical", "health",
        "pharmacy", "doctor",
    ]):
        return "health"

    if any(k in text for k in [
        "market", "supermarket", "grocery", "mall",
        "shopping", "shop",
    ]):
        return "market"

    return "commercial_service"


pois["poi_group"] = pois.apply(
    classify_poi,
    axis=1,
)

print(pois["poi_group"].value_counts())


In [ ]:
# 29. POI DENSITIES -> POI/SERVICE SCORE

pois_m = pois.to_crs(TARGET_EPSG)

poi_group_scores = []

for group in [
    "market",
    "education",
    "health",
    "commercial_service",
]:
    subset = pois_m[pois_m["poi_group"] == group]

    group_raster = rasterize(
        [(geom, 1) for geom in subset.geometry],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="float32",
        merge_alg=MergeAlg.add,
    )

    group_density_np = ndimage.uniform_filter(
        group_raster,
        size=window_cells,
        mode="constant",
        cval=0,
    )

    group_da = np_to_template(
        group_density_np,
        f"{group}_density",
    )

    group_score = robust_norm_local(group_da)
    poi_group_scores.append(group_score)

poi_score = (
    sum(poi_group_scores) / len(poi_group_scores)
).clip(0, 1).astype("float32").rename("POI_Service_Score")

print("poi         : READY")


In [ ]:
# 30. FACTOR READINESS CHECK
factors = {
    "builtup": builtup_probability,
    "nightlight": nightlight_score,
    "population": population_score,
    "road": road_score,
    "poi": poi_score,
    "topography": topography_score,
}

print("FACTOR STATUS")
print("-" * 32)

for name, value in factors.items():
    print(
        f"{name:12s}:",
        "READY" if value is not None else "MISSING",
    )

missing = [
    name
    for name, value in factors.items()
    if value is None
]

if missing:
    raise RuntimeError(
        "Missing factors: " + ", ".join(missing)
    )

print("\nALL SIX FACTORS ARE READY.")


In [ ]:
# 31. SHOW SIX FACTORS
factor_titles = {
    "builtup": "Built-up Probability",
    "nightlight": "Night-Light Score",
    "population": "Population Score",
    "road": "Road Score",
    "poi": "POI / Service Score",
    "topography": "Topography Score",
}

for name, da in factors.items():
    preview = (
        clip_bd(da)
        .coarsen(x=5, y=5, boundary="trim")
        .mean(skipna=True)
        .compute()
    )

    plt.figure(figsize=(7, 8))
    preview.plot(vmin=0, vmax=1)
    plt.title(factor_titles[name])
    plt.axis("equal")
    plt.show()


In [ ]:
# 32. MULTI-FACTOR URBAN SCORE
# First align all factors to exactly the Sentinel template grid.

aligned_factors = {}

for name, da in factors.items():
    if name in ["road", "poi", "builtup"]:
        aligned = add_rio_metadata(da)
    else:
        aligned = align_to_template(
            add_rio_metadata(da),
            "bilinear",
        )

    aligned_factors[name] = aligned


urban_score = sum(
    WEIGHTS[name] * aligned_factors[name]
    for name in WEIGHTS
).clip(0, 1).astype("float32").rename("Urban_Score")

urban_score_bd = clip_bd(urban_score)

print("Urban Score READY")
print("Weights:", WEIGHTS)


In [ ]:
# 33. SHOW URBAN SCORE
urban_preview = (
    urban_score_bd
    .coarsen(x=4, y=4, boundary="trim")
    .mean(skipna=True)
    .compute()
)

plt.figure(figsize=(8, 9))
urban_preview.plot(vmin=0, vmax=1)
plt.title("Bangladesh Multi-Factor Urban Score")
plt.axis("equal")
plt.show()


In [ ]:
# 34. URBAN CENTER MASK
urban_mask = (
    urban_score_bd >= URBAN_SCORE_THRESHOLD
).astype("uint8").rename("Urban_Center_Mask")

mask_preview = (
    urban_mask
    .coarsen(x=4, y=4, boundary="trim")
    .max()
    .compute()
)

plt.figure(figsize=(8, 9))
mask_preview.plot()
plt.title(
    f"Urban Center Candidate Mask "
    f"(Score ≥ {URBAN_SCORE_THRESHOLD})"
)
plt.axis("equal")
plt.show()


In [44]:
# 35. CONNECTED COMPONENTS + MINIMUM PATCH SIZE

mask_np = urban_mask.compute().values.astype(bool)

# 8-neighbour connectivity
structure = np.ones((3, 3), dtype="uint8")

labels, n_components = ndimage.label(
    mask_np,
    structure=structure,
)

pixel_area_km2 = (
    RESOLUTION_M * RESOLUTION_M
) / 1_000_000.0

min_pixels = max(
    1,
    int(np.ceil(
        MIN_PATCH_AREA_KM2 / pixel_area_km2
    )),
)

counts = np.bincount(labels.ravel())

keep_labels = np.where(
    counts >= min_pixels
)[0]

keep_labels = keep_labels[
    keep_labels != 0
]

filtered_np = np.isin(
    labels,
    keep_labels,
)

filtered_mask = xr.DataArray(
    filtered_np.astype("uint8"),
    dims=("y", "x"),
    coords={
        "y": urban_mask.y.values,
        "x": urban_mask.x.values,
    },
    name="Urban_Center_Filtered",
)

filtered_mask = (
    filtered_mask.rio
    .set_spatial_dims(
        x_dim="x",
        y_dim="y",
        inplace=False,
    )
    .rio.write_crs(
        f"EPSG:{TARGET_EPSG}",
        inplace=False,
    )
)

print("Initial components:", n_components)
print("Minimum patch:", MIN_PATCH_AREA_KM2, "km²")
print("Minimum pixels:", min_pixels)
print("Retained components:", len(keep_labels))


NameError: name 'urban_mask' is not defined

In [ ]:
# 36. POLYGONIZE URBAN CENTER CANDIDATES

records = []

for geom, value in shapes(
    filtered_mask.values,
    mask=filtered_mask.values.astype(bool),
    transform=filtered_mask.rio.transform(),
):
    if int(value) == 1:
        records.append({
            "geometry": shape(geom),
            "class": 1,
        })

urban_polygons = gpd.GeoDataFrame(
    records,
    crs=f"EPSG:{TARGET_EPSG}",
)

if urban_polygons.empty:
    print("No candidate polygons found.")
else:
    urban_polygons["area_km2"] = (
        urban_polygons.geometry.area
        / 1_000_000.0
    )

    urban_polygons = (
        urban_polygons
        .reset_index(drop=True)
    )

    urban_polygons["urban_id"] = np.arange(
        1,
        len(urban_polygons) + 1,
    )

    print(
        "Candidate urban-center polygons:",
        len(urban_polygons),
    )


In [ ]:
# 37. ZONAL MEAN URBAN SCORE + POPULATION DENSITY

def zonal_mean(gdf, da, field_name):
    arr = da.compute().values
    transform = da.rio.transform()

    result = []

    for geom in gdf.geometry:
        zone = rasterize(
            [(geom, 1)],
            out_shape=arr.shape,
            transform=transform,
            fill=0,
            dtype="uint8",
            all_touched=False,
        ).astype(bool)

        vals = arr[zone]
        vals = vals[np.isfinite(vals)]

        result.append(
            float(vals.mean())
            if vals.size
            else np.nan
        )

    gdf[field_name] = result
    return gdf


if not urban_polygons.empty:
    urban_polygons = zonal_mean(
        urban_polygons,
        urban_score_bd,
        "mean_score",
    )

    population_density_bd = clip_bd(
        align_to_template(
            population_density,
            "bilinear",
        )
    )

    urban_polygons = zonal_mean(
        urban_polygons,
        population_density_bd,
        "mean_popden",
    )

    display(
        urban_polygons[
            [
                "urban_id",
                "area_km2",
                "mean_score",
                "mean_popden",
            ]
        ].head(20)
    )


In [ ]:
# 38. FINAL URBAN-CENTER FILTERING
if urban_polygons.empty:
    final_urban_centers = urban_polygons.copy()
else:
    final_urban_centers = urban_polygons[
        (urban_polygons["area_km2"] >= MIN_PATCH_AREA_KM2)
        & (
            urban_polygons["mean_score"]
            >= MIN_MEAN_URBAN_SCORE
        )
        & (
            urban_polygons["mean_popden"]
            >= MIN_MEAN_POP_DENSITY
        )
    ].copy()

    final_urban_centers = (
        final_urban_centers
        .reset_index(drop=True)
    )

    final_urban_centers["urban_id"] = np.arange(
        1,
        len(final_urban_centers) + 1,
    )

print(
    "Final urban centers:",
    len(final_urban_centers),
)


In [ ]:
# 39. FINAL INTERACTIVE MAP IN VS CODE

final_map = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="750px",
)

final_map.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={
        "color": "black",
        "weight": 2,
        "fillOpacity": 0,
    },
)

if not final_urban_centers.empty:
    final_map.add_gdf(
        final_urban_centers.to_crs(4326),
        layer_name="Final Urban Centers",
        style={
            "color": "red",
            "weight": 1,
            "fillColor": "red",
            "fillOpacity": 0.50,
        },
    )

final_map


In [ ]:
# 40. SAVE ONLY FINAL / IMPORTANT OUTPUTS
#
# Source data were queried online.
# These are your analysis outputs.

outputs = {
    "Builtup_Probability_2025_100m.tif":
        clip_bd(builtup_probability),

    "NightLight_Score_2025_100m.tif":
        clip_bd(aligned_factors["nightlight"]),

    "Population_Score_2025_100m.tif":
        clip_bd(aligned_factors["population"]),

    "Road_Score_2025_100m.tif":
        clip_bd(aligned_factors["road"]),

    "POI_Service_Score_2025_100m.tif":
        clip_bd(aligned_factors["poi"]),

    "Topography_Score_100m.tif":
        clip_bd(aligned_factors["topography"]),

    "Urban_Score_2025_100m.tif":
        urban_score_bd,

    "Urban_Center_Mask_2025_100m.tif":
        filtered_mask,
}

for filename, da in outputs.items():
    path = OUTPUT_DIR / filename
    print("Writing:", path)

    da.rio.to_raster(
        path,
        compress="DEFLATE",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

gpkg_path = (
    OUTPUT_DIR
    / "Bangladesh_Urban_Centers_2025.gpkg"
)

final_urban_centers.to_file(
    gpkg_path,
    layer="urban_centers",
    driver="GPKG",
)

print("\nDONE")
print("Final vector:", gpkg_path)
print("Output folder:", OUTPUT_DIR)


## Final logic

`Sentinel built-up probability × 0.30`  
`+ VIIRS persistent-light score × 0.20`  
`+ GHSL population score × 0.20`  
`+ Overture road score × 0.10`  
`+ Overture POI/service score × 0.10`  
`+ topography score × 0.10`  
`= Urban Score (0–1)`

Then:

**Urban Score threshold → connected components → minimum area → mean score + mean population density filter → final urban-center polygons**
